# Negation Awareness CLIP - Comprehensive Experiments

Two main experiments:
1. **Pairwise Preference Evaluation** - All models on NegRefCOCOg
2. **Recall@K Evaluation** - Different models on COCOValLlama

In [1]:
import sys
import os

project_root = os.path.dirname(os.path.abspath('.'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✓ Project root: {project_root}")

import torch
import json
from pathlib import Path
from torch.utils.data import DataLoader
import clip
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

from src.data import (
    NegationJSONDataset, 
    COCOValLlamaDataset, 
    NegRefCOCOgDataset, 
    VALSEDataset
)
from src.features import FeatureCache, extract_and_cache
from src.llm import LocalQwenClient, SubQueryExtractor
from src.models import DEOModel, NegationSteeredCLIP, load_negation_direction
from src.training import train_binary_negation_classifier, steer_embeddings
from src.evaluation import PairwiseModelAdapter, evaluate_pairwise_preference, evaluate_image_text_retrieval
from src.experiments import run_paper_negation_experiment, evaluate_negation_steering_on_text

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Device: {device}")
print(f"✓ All modules imported successfully!")

✓ Project root: /home/kritic/dlcv


/home/kritic/miniconda3/envs/dlcv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Device: cuda
✓ All modules imported successfully!


In [ ]:
def get_models(negation_vector, prompt_version="v1", model_id="Qwen/Qwen2.5-1.5B-Instruct", dataset_name="NegRefCOCOg"):
    # Load base CLIP model
    print("Loading base CLIP model...")
    arch = "ViT-B/32"
    base_clip, preprocess = clip.load(arch, device=device)
    base_clip.eval()
    print(f"Base CLIP loaded to {device}")

    # Load NegationCLIP checkpoint
    print("Loading NegationCLIP checkpoint...")
    try:
        negation_clip, _ = clip.load(arch, device=device)
        checkpoint = torch.load("negationclip_ViT-B32.pth", map_location=device)
        negation_clip.load_state_dict(checkpoint)
        negation_clip.eval()
        print(f"NegationCLIP checkpoint loaded")
    except Exception as e:
        negation_clip = None
        print(f"NegationCLIP checkpoint not found: {e}")


    # Load negation vector and create steered CLIP
    print("Loading negation vector and creating Steered CLIP...")
    try:
        from pathlib import Path
        negation_vector_candidates = sorted(Path("learned_vectors").glob(negation_vector))
        if negation_vector_candidates:
            negation_vector_path = str(negation_vector_candidates[-1])
        else:
            # Fallback to any negation vector
            negation_vector_candidates = sorted(Path("learned_vectors").glob("*_negation_vector.pt"))
            negation_vector_path = str(negation_vector_candidates[-1]) if negation_vector_candidates else None
        
        if negation_vector_path:
            print(f"  Using: {Path(negation_vector_path).name}")
            w_dir = load_negation_direction(negation_vector_path)
            org_steered_clip = NegationSteeredCLIP(base_clip, w_dir=w_dir, alpha=0.13).to(device)
            org_steered_clip.eval()

            neg_steered_clip = NegationSteeredCLIP(negation_clip, w_dir=w_dir, alpha=0.13).to(device)
            neg_steered_clip.eval()
            print(f"Steered CLIP created (alpha=0.13)")
        else:
            org_steered_clip = None
            neg_steered_clip = None
            print("No negation vector found")
    except Exception as e:
        org_steered_clip = None
        neg_steered_clip = None
        print(f"Error loading negation vector: {e}")


    # Setup DEO models
    print("Setting up DEO models...")
    try:
        # Initialize LLM client for DEO
        from src.llm import LocalQwenClient
        qwen_client = LocalQwenClient(model_id=model_id, device=device)
        
        # Setup extractor
        negcocoreg_extractor = SubQueryExtractor(
            llm_client=qwen_client,
            model_name=qwen_client.model_id,
            prompt_version=prompt_version,
            dataset_name=dataset_name,
        )
        
        # DEO config for original CLIP
        deo_cfg = {
            "lr": 0.01,
            "steps": 20,
            "pos_weight": 1.0,
            "neg_weight": 1.0,
            "reg_weight": 0.2,
        }
        deo_original = DEOModel(base_clip, negcocoreg_extractor, config=deo_cfg, device=device).to(device)
        deo_original.eval()
        
        # DEO config for NegationCLIP (if available)
        if negation_clip is not None:
            deo_cfg_negation = {
                "lr": 0.01,
                "steps": 20,
                "pos_weight": 1.0,
                "neg_weight": 1.0,
                "reg_weight": 0.2,
            }
            deo_negation = DEOModel(negation_clip, negcocoreg_extractor, config=deo_cfg_negation, device=device).to(device)
            deo_negation.eval()
        else:
            deo_negation = None
        
        print(f"DEO models initialized")
    except Exception as e:
        deo_original = None
        deo_negation = None
        print(f"DEO initialization failed: {e}")

    # Setup model adapters
    print("Setting up model adapters...")
    models = {}

    # Add Baseline CLIP
    models["Original CLIP"] = PairwiseModelAdapter(
        "Original CLIP", base_clip, device=device, text_mode="tokenized"
    )

    # Add Steered CLIP if available
    if org_steered_clip is not None:
        models["Org + Steered CLIP"] = PairwiseModelAdapter(
            "Org + Steered CLIP", org_steered_clip, device=device, text_mode="tokenized"
        )

    if neg_steered_clip is not None:
        models["Neg + Steered CLIP"] = PairwiseModelAdapter(
            "Neg + Steered CLIP", neg_steered_clip, device=device, text_mode="tokenized"
        )

    # Add NegationCLIP if available
    if negation_clip is not None:
        models["NegationCLIP"] = PairwiseModelAdapter(
            "NegationCLIP", negation_clip, device=device, text_mode="tokenized"
        )

    # Add DEO + Original CLIP if available
    if deo_original is not None:
        models["DEO + Original CLIP"] = PairwiseModelAdapter(
            "DEO + Original CLIP", deo_original, device=device, text_mode="raw"
        )

    # Add DEO + NegationCLIP if available
    if deo_negation is not None:
        models["DEO + NegationCLIP"] = PairwiseModelAdapter(
            "DEO + NegationCLIP", deo_negation, device=device, text_mode="raw"
        )

    print(f"{len(models)} models ready for evaluation")
    return models, preprocess


## Experiment 1: Pairwise Preference Evaluation (NegRefCOCOg)

Evaluate all models on the NegRefCOCOg dataset using pairwise preference metric:
- Score = 1 if sim(text, positive_image) > sim(text, negative_image)
- Tests: Baseline CLIP, Steered CLIP, DEO variants

In [ ]:
def experiment_no_1(learned_vectors, dataloader, device, prompt_version="v1", model_id="Qwen/Qwen2.5-1.5B-Instruct"):
    # ============================================================================
    # EXPERIMENT 1: Pairwise Preference on NegRefCOCOg (Model Comparison)
    # ============================================================================
    print("\n" + "="*70)
    print("EXPERIMENT 1: PAIRWISE PREFERENCE EVALUATION (NegRefCOCOg)")
    print("="*70)

    # Evaluate all models
    print("\n" + "="*70)
    print("EVALUATING MODELS")
    print("="*70)
    
    models, preprocess = get_models(learned_vectors, prompt_version=prompt_version, model_id=model_id)
    pairwise_results = {}
    for i, (model_name, adapter) in enumerate(models.items(), 1):
        print(f"\nModel {i}/{len(models)}: {model_name}...")
        try:
            result = evaluate_pairwise_preference(
                model_adapter=adapter,
                dataset_loader=dataloader,
                preprocess=preprocess,
                device=device,
            )
            pairwise_results[model_name] = result
            print(f"Score: {result['avg_score']:.4f} ({result['correct']}/{result['total']})")
        except Exception as e:
            print(f"Error evaluating {model_name}: {e}")

    # Print Summary
    print("\n" + "="*70)
    print("PAIRWISE PREFERENCE RESULTS SUMMARY (NegRefCOCOg)")
    print("="*70)
    print(f"{'Model':<30} {'Score':<12} {'Accuracy':<12} {'Count':<12}")
    print("-"*70)

    for model_name, result in pairwise_results.items():
        acc = result['avg_score']
        count = f"{result['correct']}/{result['total']}"
        print(f"{model_name:<30} {result['avg_score']:.4f}      {acc:.2%}        {count}")

    if pairwise_results:
        best_model = max(pairwise_results.items(), key=lambda x: x[1]['avg_score'])
        print(f"Best Model: {best_model[0]} with score {best_model[1]['avg_score']:.4f}")

    print("="*70)
    return pairwise_results

In [ ]:
def experiment_no_2(learned_vectors, dataloader, device, prompt_version='v1', model_id="Qwen/Qwen2.5-1.5B-Instruct", dataset_name="VALSE"):
    # ============================================================================
    # EXPERIMENT 1: Pairwise Preference on NegRefCOCOg (Model Comparison)
    # ============================================================================
    print("\n" + "="*70)
    print("EXPERIMENT 1: PAIRWISE PREFERENCE EVALUATION (NegRefCOCOg)")
    print("="*70)

    # Evaluate all models
    print("\n" + "="*70)
    print("EVALUATING MODELS")
    print("="*70)
    
    models, preprocess = get_models(learned_vectors, prompt_version=prompt_version, model_id=model_id, dataset_name=dataset_name)
    pairwise_results = {}
    for i, (model_name, adapter) in enumerate(models.items(), 1):
        print(f"Model {i}/{len(models)}: {model_name}...")
        try:
            result = evaluate_image_text_retrieval(
                model_adapter=adapter,
                dataset_loader=dataloader,
                device=device,
            )
            pairwise_results[model_name] = result
            print(f"Score: {result['avg_score']:.4f} ({result['correct']}/{result['total']})")
        except Exception as e:
            print(f"Error evaluating {model_name}: {e}")

    # Print Summary
    print("\n" + "="*70)
    print("PAIRWISE PREFERENCE RESULTS SUMMARY (NegRefCOCOg)")
    print("="*70)
    print(f"{'Model':<30} {'Score':<12} {'Accuracy':<12} {'Count':<12}")
    print("-"*70)

    for model_name, result in pairwise_results.items():
        acc = result['avg_score']
        count = f"{result['correct']}/{result['total']}"
        print(f"{model_name:<30} {result['avg_score']:.4f}      {acc:.2%}        {count}")

    if pairwise_results:
        best_model = max(pairwise_results.items(), key=lambda x: x[1]['avg_score'])
        print(f"Best Model: {best_model[0]} with score {best_model[1]['avg_score']:.4f}")

    print("="*70)
    return pairwise_results

In [ ]:
# Dataset setup
print("Loading NegRefCOCOg dataset...")
negref_dataset = NegRefCOCOgDataset("./data/NegRefCOCOg.json", max_samples=7000)
negref_loader = DataLoader(negref_dataset, batch_size=32, shuffle=False)
print(f"Loaded {len(negref_dataset)} samples")


[1/6] Loading NegRefCOCOg dataset...
✓ Loaded 440 samples


In [ ]:
valse_dataset = VALSEDataset("./data/existence.json", max_samples=7000)
valse_loader = DataLoader(valse_dataset, batch_size=32, shuffle=False)
print(f"Loaded {len(valse_dataset)} samples")

✓ Loaded 534 samples


<h2>On NegRefCOCOg Dataset

In [25]:
learned_vectors = "GEMINI_NEGATION_2_Baseline_CLIP_Baseline_CLIP_8dbd35bd_negation_vector.pt"
prompt_version = "v1"
model_id = "Qwen/Qwen3-4B-Instruct-2507"  # Use text-only Qwen for LLM client (not multimodal)
print ("Running Experiment 1 with learned vectors from original clip layer 5", learned_vectors)
results_1 = experiment_no_1(learned_vectors, negref_loader, device, prompt_version=prompt_version, model_id=model_id)

Running Experiment 1 with learned vectors from original clip layer 5 GEMINI_NEGATION_2_Baseline_CLIP_Baseline_CLIP_8dbd35bd_negation_vector.pt

EXPERIMENT 1: PAIRWISE PREFERENCE EVALUATION (NegRefCOCOg)

EVALUATING MODELS

[2/6] Loading base CLIP model...
✓ Base CLIP loaded to cuda

[4/6] Loading NegationCLIP checkpoint...


/tmp/ipykernel_710863/913513589.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("negationclip_ViT-B32.pth", map_location=device)


✓ NegationCLIP checkpoint loaded

[3/6] Loading negation vector and creating Steered CLIP...
  Using: GEMINI_NEGATION_2_Baseline_CLIP_Baseline_CLIP_8dbd35bd_negation_vector.pt
✓ Steered CLIP created (alpha=0.13)

[5/6] Setting up DEO models...


Loading weights: 100%|██████████| 398/398 [00:02<00:00, 172.87it/s]


✓ DEO models initialized

[6/6] Setting up model adapters...
✓ 6 models ready for evaluation

Model 1/6: Original CLIP...


...

✓ Score: 0.6341 (279/440)

Model 5/6: DEO + Original CLIP...


Evaluating DEO + Original CLIP: 100%|██████████| 14/14 [00:29<00:00,  2.13s/it]


✓ Score: 0.5886 (259/440)

Model 6/6: DEO + NegationCLIP...


Evaluating DEO + NegationCLIP: 100%|██████████| 14/14 [00:29<00:00,  2.12s/it]

✓ Score: 0.6591 (290/440)

PAIRWISE PREFERENCE RESULTS SUMMARY (NegRefCOCOg)
Model                          Score        Accuracy     Count       
----------------------------------------------------------------------
Original CLIP                  0.5841      58.41%        257/440
Org + Steered CLIP             0.5705      57.05%        251/440
Neg + Steered CLIP             0.6455      64.55%        284/440
NegationCLIP                   0.6341      63.41%        279/440
DEO + Original CLIP            0.5886      58.86%        259/440
DEO + NegationCLIP             0.6591      65.91%        290/440

✓ Best Model: DEO + NegationCLIP with score 0.6591


In [ ]:
learned_vectors = "GEMINI_NEGATION_2_NG_CLIP_NG_CLIP_13d418c9_negation_vector.pt"
prompt_version = "v1"
model_id = "Qwen/Qwen3-4B-Instruct-2507"  # Use text-only Qwen for LLM client (not multimodal)
print ("Running Experiment 1 with learned vectors from original clip layer 5", learned_vectors)
results_2 = experiment_no_1(learned_vectors, negref_loader, device, prompt_version=prompt_version, model_id=model_id)

Running Experiment 1 with learned vectors from original clip layer 5 GEMINI_NEGATION_2_NG_CLIP_NG_CLIP_13d418c9_negation_vector.pt

EXPERIMENT 1: PAIRWISE PREFERENCE EVALUATION (NegRefCOCOg)

EVALUATING MODELS

[2/6] Loading base CLIP model...
✓ Base CLIP loaded to cuda

[4/6] Loading NegationCLIP checkpoint...


/tmp/ipykernel_710863/913513589.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("negationclip_ViT-B32.pth", map_location=device)


✓ NegationCLIP checkpoint loaded

[3/6] Loading negation vector and creating Steered CLIP...
  Using: GEMINI_NEGATION_2_NG_CLIP_NG_CLIP_13d418c9_negation_vector.pt
✓ Steered CLIP created (alpha=0.13)

[5/6] Setting up DEO models...


/home/kritic/dlcv/Negation_Awareness_CLIP/src/models/steered_clip.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = torch.load(vector_path, map_location="cpu")


✓ DEO models initialized

[6/6] Setting up model adapters...
✓ 6 models ready for evaluation

Model 1/6: Original CLIP...


...

✓ Score: 0.6341 (279/440)

Model 5/6: DEO + Original CLIP...


Evaluating DEO + Original CLIP: 100%|██████████| 14/14 [00:30<00:00,  2.18s/it]


✓ Score: 0.5886 (259/440)

Model 6/6: DEO + NegationCLIP...


Evaluating DEO + NegationCLIP: 100%|██████████| 14/14 [00:29<00:00,  2.12s/it]

✓ Score: 0.6591 (290/440)

PAIRWISE PREFERENCE RESULTS SUMMARY (NegRefCOCOg)
Model                          Score        Accuracy     Count       
----------------------------------------------------------------------
Original CLIP                  0.5841      58.41%        257/440
Org + Steered CLIP             0.5727      57.27%        252/440
Neg + Steered CLIP             0.6477      64.77%        285/440
NegationCLIP                   0.6341      63.41%        279/440
DEO + Original CLIP            0.5886      58.86%        259/440
DEO + NegationCLIP             0.6591      65.91%        290/440

✓ Best Model: DEO + NegationCLIP with score 0.6591


<h2>On VALSE dataset

In [27]:
learned_vectors = "GEMINI_NEGATION_2_Baseline_CLIP_Baseline_CLIP_8dbd35bd_negation_vector.pt"
prompt_version = "v1"
model_id = "Qwen/Qwen3-4B-Instruct-2507" 
dataset_name = "VALSE"
print ("Running Experiment 2 with learned vectors from original clip layer 5", learned_vectors)
results_3 = experiment_no_2(learned_vectors, valse_loader, device, prompt_version=prompt_version, model_id=model_id, dataset_name=dataset_name)

Running Experiment 2 with learned vectors from original clip layer 5 GEMINI_NEGATION_2_Baseline_CLIP_Baseline_CLIP_8dbd35bd_negation_vector.pt

EXPERIMENT 1: PAIRWISE PREFERENCE EVALUATION (NegRefCOCOg)

EVALUATING MODELS

[2/6] Loading base CLIP model...
✓ Base CLIP loaded to cuda

[4/6] Loading NegationCLIP checkpoint...


/tmp/ipykernel_710863/913513589.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("negationclip_ViT-B32.pth", map_location=device)


✓ NegationCLIP checkpoint loaded

[3/6] Loading negation vector and creating Steered CLIP...
  Using: GEMINI_NEGATION_2_Baseline_CLIP_Baseline_CLIP_8dbd35bd_negation_vector.pt
✓ Steered CLIP created (alpha=0.13)

[5/6] Setting up DEO models...


Loading weights: 100%|██████████| 398/398 [00:02<00:00, 172.14it/s]


✓ DEO models initialized

[6/6] Setting up model adapters...
✓ 6 models ready for evaluation

Model 1/6: Original CLIP...


...

✓ Score: 0.7772 (415/534)

Model 5/6: DEO + Original CLIP...


Evaluating DEO + Original CLIP (Image-Text Retrieval):  82%|████████▏ | 14/17 [00:40<00:08,  2.87s/it]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Evaluating DEO + Original CLIP (Image-Text Retrieval): 100%|██████████| 17/17 [02:39<00:00,  9.40s/it]


✓ Score: 0.6742 (360/534)

Model 6/6: DEO + NegationCLIP...


Evaluating DEO + NegationCLIP (Image-Text Retrieval): 100%|██████████| 17/17 [00:50<00:00,  2.95s/it]

✓ Score: 0.7472 (399/534)

PAIRWISE PREFERENCE RESULTS SUMMARY (NegRefCOCOg)
Model                          Score        Accuracy     Count       
----------------------------------------------------------------------
Original CLIP                  0.7097      70.97%        379/534
Org + Steered CLIP             0.7060      70.60%        377/534
Neg + Steered CLIP             0.7734      77.34%        413/534
NegationCLIP                   0.7772      77.72%        415/534
DEO + Original CLIP            0.6742      67.42%        360/534
DEO + NegationCLIP             0.7472      74.72%        399/534

✓ Best Model: NegationCLIP with score 0.7772


In [29]:
learned_vectors = "GEMINI_NEGATION_2_Baseline_CLIP_Baseline_CLIP_8dbd35bd_negation_vector.pt"
prompt_version = "v1"
model_id = "Qwen/Qwen3-4B-Instruct-2507" 
dataset_name = "VALSE"
print ("Running Experiment 2 with learned vectors from original clip layer 5", learned_vectors)
results_4 = experiment_no_2(learned_vectors, valse_loader, device, prompt_version=prompt_version, model_id=model_id, dataset_name=dataset_name)

Running Experiment 2 with learned vectors from original clip layer 5 GEMINI_NEGATION_2_Baseline_CLIP_Baseline_CLIP_8dbd35bd_negation_vector.pt

EXPERIMENT 1: PAIRWISE PREFERENCE EVALUATION (NegRefCOCOg)

EVALUATING MODELS

[2/6] Loading base CLIP model...
✓ Base CLIP loaded to cuda

[4/6] Loading NegationCLIP checkpoint...


/tmp/ipykernel_710863/913513589.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("negationclip_ViT-B32.pth", map_location=device)


✓ NegationCLIP checkpoint loaded

[3/6] Loading negation vector and creating Steered CLIP...
  Using: GEMINI_NEGATION_2_Baseline_CLIP_Baseline_CLIP_8dbd35bd_negation_vector.pt
✓ Steered CLIP created (alpha=0.13)

[5/6] Setting up DEO models...


Loading weights: 100%|██████████| 398/398 [00:02<00:00, 173.79it/s]


✓ DEO models initialized

[6/6] Setting up model adapters...
✓ 6 models ready for evaluation

Model 1/6: Original CLIP...


...

✓ Score: 0.7772 (415/534)

Model 5/6: DEO + Original CLIP...


Evaluating DEO + Original CLIP (Image-Text Retrieval): 100%|██████████| 17/17 [00:48<00:00,  2.82s/it]


✓ Score: 0.6742 (360/534)

Model 6/6: DEO + NegationCLIP...


Evaluating DEO + NegationCLIP (Image-Text Retrieval): 100%|██████████| 17/17 [00:48<00:00,  2.88s/it]

✓ Score: 0.7472 (399/534)

PAIRWISE PREFERENCE RESULTS SUMMARY (NegRefCOCOg)
Model                          Score        Accuracy     Count       
----------------------------------------------------------------------
Original CLIP                  0.7097      70.97%        379/534
Org + Steered CLIP             0.7060      70.60%        377/534
Neg + Steered CLIP             0.7734      77.34%        413/534
NegationCLIP                   0.7772      77.72%        415/534
DEO + Original CLIP            0.6742      67.42%        360/534
DEO + NegationCLIP             0.7472      74.72%        399/534

✓ Best Model: NegationCLIP with score 0.7772


In [4]:
import os
import torch
import torch.nn.functional as F
from tqdm import tqdm
from pathlib import Path

# If not already imported, import the ImageNetDataset
from src.data import ImageNetDataset

def experiment_no_3_tiny_imagenet(imagenet_root, learned_vectors, device, prompt_version="v1", model_id="Qwen/Qwen2.5-1.5B-Instruct"):
    print("\n" + "="*70)
    print("EXPERIMENT 3: ZERO-SHOT CLASSIFICATION ON TINY IMAGENET")
    print("="*70)
    
    # 1. Load Dataset
    print(f"\n[1/4] Loading Tiny ImageNet dataset from {imagenet_root}...")
    tiny_imagenet = ImageNetDataset(
        root_dir=imagenet_root,
        split='val',
        max_samples=None,
        shuffle=False
    )
    classnames = tiny_imagenet.get_all_classnames()
    
    # 2. Get Models
    print("\n[2/4] Setting up models...")
    models, preprocess = get_models(
        negation_vector=learned_vectors, 
        prompt_version=prompt_version, 
        model_id=model_id, 
        dataset_name="TinyImageNet"
    )
    
    cache_dir = Path("embeddings_cache")
    cache_dir.mkdir(parents=True, exist_ok=True)
    
    results = {}
    
    print("\n[3/4] Running Zero-Shot Classification...")
    for model_name, model_adapter in models.items():
        print(f"\n{'-'*50}\nEvaluating Model: {model_name}\n{'-'*50}")
        model_adapter.model.eval()
        
        # Step A: Compute Text Embeddings for each class
        print(f"  Pre-computing text embeddings for {tiny_imagenet.num_classes()} classes...")
        text_embeddings = []
        with torch.no_grad():
            for class_name in classnames:
                prompt = f"a photo of {class_name}"
                text_emb = model_adapter.encode_text(prompt)
                text_embeddings.append(text_emb)
            
            # (num_classes, embedding_dim)
            text_embeddings = torch.cat(text_embeddings, dim=0).to(device)
            text_embeddings = F.normalize(text_embeddings, p=2, dim=-1)
            
        # Step B: Get or Cache Image Embeddings
        safe_model_name = model_name.replace(" ", "_").replace("+", "plus")
        cache_path = cache_dir / f"{safe_model_name}_tiny_imagenet_val_embeddings.pt"
        
        if cache_path.exists():
            print(f"  Loading cached image embeddings from {cache_path}...")
            cache_data = torch.load(cache_path, map_location=device)
            image_embeddings = cache_data['embeddings'].to(device)
            true_labels = cache_data['labels'].to(device)
        else:
            print("  Computing and caching image embeddings (this may take a while)...")
            image_embeddings = []
            true_labels = []
            
            with torch.no_grad():
                for i in tqdm(range(len(tiny_imagenet)), desc=f"  Encoding Images ({model_name})"):
                    sample = tiny_imagenet[i]
                    image_tensor = sample['image'].unsqueeze(0).to(device)
                    true_class_idx = sample['class_idx']
                    
                    img_emb = model_adapter.encode_image(image_tensor)
                    img_emb = F.normalize(img_emb, p=2, dim=-1)
                    
                    image_embeddings.append(img_emb.cpu())
                    true_labels.append(true_class_idx)
                    
            image_embeddings = torch.cat(image_embeddings, dim=0).to(device)
            true_labels = torch.tensor(true_labels).to(device)
            
            torch.save({
                'embeddings': image_embeddings.cpu(),
                'labels': true_labels.cpu()
            }, cache_path)
            print(f"  ✓ Cached {len(true_labels)} image embeddings.")

        # Step C: Compute Similarities and Metrics
        print("  Computing similarities and evaluation metrics...")
        
        # similarities shape: (num_images, num_classes)
        similarities = image_embeddings @ text_embeddings.T
        
        # Get Top-5 predictions
        top5_preds = torch.topk(similarities, k=5, dim=-1).indices
        top1_preds = top5_preds[:, 0]
        
        # Calculate Top-1 Accuracy
        correct_top1 = (top1_preds == true_labels).sum().item()
        
        # Calculate Top-5 Accuracy
        correct_top5 = 0
        for i in range(len(true_labels)):
            if true_labels[i] in top5_preds[i]:
                correct_top5 += 1
                
        total = len(true_labels)
        top1_acc = correct_top1 / max(total, 1)
        top5_acc = correct_top5 / max(total, 1)
        
        print(f"  ✓ Top-1 Accuracy: {top1_acc:.4f} ({correct_top1}/{total})")
        print(f"  ✓ Top-5 Accuracy: {top5_acc:.4f} ({correct_top5}/{total})")
        
        results[model_name] = {
            'top1': top1_acc,
            'top5': top5_acc
        }

    # 4. Final Summary
    print("\n[4/4] Final Results Summary")
    print("="*70)
    print(f"{'Model':<30} {'Top-1 Accuracy':<15} {'Top-5 Accuracy':<15}")
    print("-" * 70)
    for model_name, metrics in results.items():
        print(f"{model_name:<30} {metrics['top1']:.4f}          {metrics['top5']:.4f}")
    print("="*70)
    
    return results


In [ ]:

# ==========================================
# Run the experiment 
# ==========================================
# Make sure to update the imagenet_root path to your actual local dataset directory
imagenet_root = "./imagenet" 
learned_vectors = "GEMINI_NEGATION_2_Baseline_CLIP_Baseline_CLIP_8dbd35bd_negation_vector.pt"
prompt_version = "v1"
model_id = "Qwen/Qwen3-4B-Instruct-2507"

# Execute the experiment
tiny_imagenet_results = experiment_no_3_tiny_imagenet(
    imagenet_root=imagenet_root,
    learned_vectors=learned_vectors,
    device=device,
    prompt_version=prompt_version,
    model_id=model_id
)

#GEMINI_NEGATION_2_NG_CLIP_NG_CLIP_13d418c9_negation_vector


EXPERIMENT 3: ZERO-SHOT CLASSIFICATION ON TINY IMAGENET

[1/4] Loading Tiny ImageNet dataset from ./imagenet...
✓ Detected Tiny ImageNet structure
✓ Tiny ImageNet val: 10000 images, 200 classes

[2/4] Setting up models...

[2/6] Loading base CLIP model...
✓ Base CLIP loaded to cuda

[4/6] Loading NegationCLIP checkpoint...


/tmp/ipykernel_710863/913513589.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("negationclip_ViT-B32.pth", map_location=device)


✓ NegationCLIP checkpoint loaded

[3/6] Loading negation vector and creating Steered CLIP...
  Using: GEMINI_NEGATION_2_Baseline_CLIP_Baseline_CLIP_8dbd35bd_negation_vector.pt
✓ Steered CLIP created (alpha=0.13)

[5/6] Setting up DEO models...


Loading weights: 100%|██████████| 398/398 [00:02<00:00, 170.91it/s]


✓ DEO models initialized

[6/6] Setting up model adapters...
✓ 6 models ready for evaluation

[3/4] Running Zero-Shot Classification...

--------------------------------------------------
Evaluating Model: Original CLIP
--------------------------------------------------
  Pre-computing text embeddings for 200 classes...
  Computing and caching image embeddings (this may take a while)...


...

  ✓ Cached 10000 image embeddings.
  Computing similarities and evaluation metrics...
  ✓ Top-1 Accuracy: 0.5397 (5397/10000)
  ✓ Top-5 Accuracy: 0.7653 (7653/10000)

--------------------------------------------------
Evaluating Model: DEO + Original CLIP
--------------------------------------------------
  Pre-computing text embeddings for 200 classes...
  Computing and caching image embeddings (this may take a while)...


  Encoding Images (DEO + Original CLIP): 100%|██████████| 10000/10000 [01:22<00:00, 120.72it/s]


  ✓ Cached 10000 image embeddings.
  Computing similarities and evaluation metrics...
  ✓ Top-1 Accuracy: 0.5518 (5518/10000)
  ✓ Top-5 Accuracy: 0.7894 (7894/10000)

--------------------------------------------------
Evaluating Model: DEO + NegationCLIP
--------------------------------------------------
  Pre-computing text embeddings for 200 classes...
  Computing and caching image embeddings (this may take a while)...


  Encoding Images (DEO + NegationCLIP): 100%|██████████| 10000/10000 [01:34<00:00, 106.30it/s]


  ✓ Cached 10000 image embeddings.
  Computing similarities and evaluation metrics...
  ✓ Top-1 Accuracy: 0.5466 (5466/10000)
  ✓ Top-5 Accuracy: 0.7803 (7803/10000)

[4/4] Final Results Summary
Model                          Top-1 Accuracy  Top-5 Accuracy 
----------------------------------------------------------------------
Original CLIP                  0.5288          0.7597
Org + Steered CLIP             0.5290          0.7586
Neg + Steered CLIP             0.5385          0.7658
NegationCLIP                   0.5397          0.7653
DEO + Original CLIP            0.5518          0.7894
DEO + NegationCLIP             0.5466          0.7803


In [5]:
imagenet_root = "./imagenet" 
learned_vectors = "GEMINI_NEGATION_2_NG_CLIP_NG_CLIP_13d418c9_negation_vector.pt"
prompt_version = "v1"
model_id = "Qwen/Qwen3-4B-Instruct-2507"

# Execute the experiment
tiny_imagenet_results = experiment_no_3_tiny_imagenet(
    imagenet_root=imagenet_root,
    learned_vectors=learned_vectors,
    device=device,
    prompt_version=prompt_version,
    model_id=model_id
)


EXPERIMENT 3: ZERO-SHOT CLASSIFICATION ON TINY IMAGENET

[1/4] Loading Tiny ImageNet dataset from ./imagenet...
✓ Detected Tiny ImageNet structure
✓ Tiny ImageNet val: 10000 images, 200 classes

[2/4] Setting up models...

[2/6] Loading base CLIP model...
✓ Base CLIP loaded to cuda

[4/6] Loading NegationCLIP checkpoint...


/tmp/ipykernel_718286/913513589.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("negationclip_ViT-B32.pth", map_location=device)


✓ NegationCLIP checkpoint loaded

[3/6] Loading negation vector and creating Steered CLIP...
  Using: GEMINI_NEGATION_2_NG_CLIP_NG_CLIP_13d418c9_negation_vector.pt
✓ Steered CLIP created (alpha=0.13)

[5/6] Setting up DEO models...


/home/kritic/dlcv/Negation_Awareness_CLIP/src/models/steered_clip.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = torch.load(vector_path, map_location="cpu")


✓ DEO models initialized

[6/6] Setting up model adapters...
✓ 6 models ready for evaluation

[3/4] Running Zero-Shot Classification...

--------------------------------------------------
Evaluating Model: Original CLIP
--------------------------------------------------
  Pre-computing text embeddings for 200 classes...
  Loading cached image embeddings from embeddings_cache/Original_CLIP_tiny_imagenet_val_embeddings.pt...
  Computing similarities and evaluation metrics...


/tmp/ipykernel_718286/1647391671.py:63: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cache_data = torch.load(cache_path, map_location=device)


  ✓ Top-1 Accuracy: 0.5288 (5288/10000)
  ✓ Top-5 Accuracy: 0.7597 (7597/10000)

--------------------------------------------------
Evaluating Model: Org + Steered CLIP
--------------------------------------------------
  Pre-computing text embeddings for 200 classes...
  Loading cached image embeddings from embeddings_cache/Org_plus_Steered_CLIP_tiny_imagenet_val_embeddings.pt...
  Computing similarities and evaluation metrics...
  ✓ Top-1 Accuracy: 0.5295 (5295/10000)
  ✓ Top-5 Accuracy: 0.7585 (7585/10000)

--------------------------------------------------
Evaluating Model: Neg + Steered CLIP
--------------------------------------------------
  Pre-computing text embeddings for 200 classes...
  Loading cached image embeddings from embeddings_cache/Neg_plus_Steered_CLIP_tiny_imagenet_val_embeddings.pt...
  Computing similarities and evaluation metrics...
  ✓ Top-1 Accuracy: 0.5392 (5392/10000)
  ✓ Top-5 Accuracy: 0.7666 (7666/10000)

-------------------------------------------------

## Experiment 2: Recall@K Evaluation (COCOValLlama)

Evaluate models on COCOValLlama using Recall@K metric:
- For each text query, rank all images by similarity
- Recall@K = fraction of queries where positive image is in top-K

In [8]:
# ============================================================================
# EXPERIMENT 2: Negation-Aware Image Retrieval + MLLM-as-Judge Evaluation
# ============================================================================
# Based on "When Negation Is a Geometry Problem in Vision-Language Models"
# 
# Pipeline:
# 1. Load negated queries from CSV dataset
# 2. Encode all COCO images with CLIP (cache embeddings)
# 3. For each query: retrieve top-5 images by similarity
# 4. For each retrieved image: Use vision-language model to judge:
#    - qr: Is image contextually correct? (ground-truth: yes)
#    - qn: Are negated objects absent? (ground-truth: yes)
# 5. Compute metrics: Top-1, Avg-5, Top-5 (both qr-only and joint qr+qn)

import random

print("\n" + "="*70)
print("EXPERIMENT 2: NEGATION-AWARE IMAGE RETRIEVAL + MLLM JUDGE")
print("="*70)

# Step 1: Load CSV Dataset
print("\n[1/6] Loading negated retrieval CSV dataset...")
from src.data import NegatedRetrievalCSVDataset
csv_dataset = NegatedRetrievalCSVDataset(
    "COCO_val_negated_retrieval_llama3.1_rephrased_affneg_true.csv",
    max_samples=100,  # Use 1000 samples for evaluation
    shuffle=True
)
print(f"✓ Loaded {len(csv_dataset)} samples")

# Step 2: Load CLIP model and prepare image embeddings
print("\n[2/6] Loading CLIP model for image ranking...")
ranking_clip, ranking_preprocess = clip.load("ViT-B/32", device=device)
ranking_clip.eval()
print(f"✓ CLIP ViT-B/32 loaded to {device}")

# Step 3: Load all COCO val images and compute embeddings
print("\n[3/6] Loading and embedding COCO val2017 images...")
from pathlib import Path
from PIL import Image

coco_images_dir = Path("val2017")
coco_image_files = sorted(list(coco_images_dir.glob("*.jpg")))
print(f"✓ Found {len(coco_image_files)} COCO images")

# Cache image embeddings with descriptive filename
# MODIFICATION 1: Cache filename includes model name, architecture, and dataset
model_name = "CLIP"
model_arch = "ViT-B32"
dataset_name = "COCO_val2017"
cache_filename = f"{model_name}_{model_arch}_{dataset_name}_embeddings.pt"
cache_path = Path("embeddings_cache") / cache_filename

if cache_path.exists():
    print(f"Loading cached embeddings from {cache_path}...")
    embeddings_data = torch.load(cache_path)
    image_embeddings = embeddings_data['embeddings']
    image_ids = embeddings_data['image_ids']
    print(f"✓ Loaded cached embeddings for {len(image_ids)} images")
else:
    print("Computing image embeddings (this may take a while)...")
    image_embeddings = []
    image_ids = []
    
    with torch.no_grad():
        for idx, img_path in enumerate(coco_image_files):
            if idx % 500 == 0:
                print(f"  Progress: {idx}/{len(coco_image_files)}")
            
            try:
                img = Image.open(img_path).convert('RGB')
                img_tensor = ranking_preprocess(img).unsqueeze(0).to(device)
                img_emb = ranking_clip.encode_image(img_tensor)
                img_emb = torch.nn.functional.normalize(img_emb, p=2, dim=-1)
                image_embeddings.append(img_emb.cpu())
                image_ids.append(int(img_path.stem))
            except Exception as e:
                print(f"  Warning: Could not load {img_path}: {e}")
                continue
    
    image_embeddings = torch.cat(image_embeddings, dim=0)
    
    # Cache embeddings
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        'embeddings': image_embeddings,
        'image_ids': image_ids
    }, cache_path)
    print(f"✓ Computed and cached {len(image_ids)} image embeddings to {cache_filename}")

# Step 4: Initialize Lightweight MLLM Judge
# Use Qwen2-VL-2B for better discriminative quality than LLaVA-Phi
print("\n[4/6] Loading Qwen2-VL-2B MLLM Judge...")
try:
    from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
    import re
    
    print("  Loading Qwen/Qwen2-VL-2B-Instruct (better discriminative power, ~2B parameters)...")
    # Qwen2-VL: ~2B parameters, more discriminative than LLaVA-Phi
    mllm_model = Qwen2VLForConditionalGeneration.from_pretrained(
        "Qwen/Qwen2-VL-2B-Instruct",
        torch_dtype=torch.float16,
        device_map=device,
        trust_remote_code=True
    )
    mllm_processor = AutoProcessor.from_pretrained(
        "Qwen/Qwen2-VL-2B-Instruct",
        trust_remote_code=True
    )
    mllm_available = True
    print(f"✓ Qwen2-VL-2B MLLM Judge loaded (~2B parameters, better discriminative quality)")
except Exception as e:
    print(f"⚠ MLLM not available: {e}")
    print(f"⚠ Will use simulated judge responses for now")
    mllm_available = False

# Step 5: Evaluation Pipeline
print("\n[5/6] Setting up evaluation metrics...")

def extract_yes_no_response(response_text):
    """Extract yes/no answer from MLLM response - more robust parsing"""
    response_lower = response_text.lower().strip()
    
    # Remove common prefixes from MLLM responses
    response_lower = response_lower.replace('answer:', '').replace('response:', '').strip()
    
    # Check for explicit yes/no at the start (more discriminative)
    if response_lower.startswith('yes'):
        return True
    elif response_lower.startswith('no'):
        return False
    # If not at start, check if it contains yes or no
    elif 'yes' in response_lower:
        return True
    elif 'no' in response_lower:
        return False
    else:
        # Default: if unclear, say "no" to be conservative (not permissive)
        return False

def evaluate_with_mllm(image_path, pos_objects, neg_objects, caption, question_type='qr'):
    """
    Use MLLM to evaluate image against negation query.
    question_type: 'qr' (semantic) or 'qn' (negation)
    """
    try:
        if not mllm_available:
            # Simulate MLLM responses
            if question_type == 'qr':
                return random.uniform(0.7, 0.95)  # Higher confidence for semantic
            else:
                return random.uniform(0.6, 0.9)   # Slightly lower for negation
        
        # Load image
        img = Image.open(image_path).convert('RGB')
        
        if question_type == 'qr':
            # Question qr: Semantic correctness - MUST check BOTH positive AND negative
            prompt = f"Does this image contain these objects: {', '.join(pos_objects)} but does NOT contain: {', '.join(neg_objects)}? Answer only 'yes' or 'no'."
        else:
            # Question qn: Negation verification - check negated objects are absent
            prompt = f"Verify that NONE of these objects appear in the image: {', '.join(neg_objects)}. Answer only 'yes' or 'no'."
        
        # Prepare inputs
        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": prompt}
                ]
            }
        ]
        
        # Process and generate
        text_prompt = mllm_processor.apply_chat_template(conversation, add_generation_prompt=True)
        inputs = mllm_processor(text=text_prompt, images=[img], return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            output_ids = mllm_model.generate(**inputs, max_new_tokens=50)
        
        # IMPORTANT: Extract only the NEW generated tokens, not the input
        # Get the number of input tokens to skip them
        input_token_length = inputs['input_ids'].shape[1]
        generated_token_ids = output_ids[0, input_token_length:]
        
        # Decode only the generated tokens
        response = mllm_processor.decode(generated_token_ids, skip_special_tokens=True)
        is_correct = extract_yes_no_response(response)
        return 1.0 if is_correct else 0.0
        
    except Exception as e:
        print(f"    Error evaluating {image_path}: {e}")
        return 0.5  # Uncertain response

# Step 6: Run evaluation on all models
print("\n[6/6] Running retrieval + MLLM evaluation...")

# Create models dict for Experiment 2 if not already defined (from Experiment 1)
# This makes Experiment 2 self-contained and can run independently
if 'models' not in locals():
    print("⚠ Note: Experiment 1 not executed. Creating minimal models dict for Experiment 2...")
    from src.evaluation import PairwiseModelAdapter
    models = {
        "CLIP + MLLM Judge": PairwiseModelAdapter(
            "CLIP + MLLM Judge", ranking_clip, device=device, text_mode="tokenized"
        )
    }
    print(f"✓ Created {len(models)} model(s) for evaluation")

evaluation_metrics = {}

for model_name, adapter in models.items():
    print(f"\n{'='*70}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*70}")
    
    # Initialize metrics
    metrics = {
        'top1_qr': 0,        # Top-1 semantic correct
        'top5_qr': 0,        # At least one in top-5 semantic correct
        'avg5_qr': 0,        # Average semantic correctness over top-5
        'top1_joint': 0,     # Top-1 both qr and qn correct
        'top5_joint': 0,     # At least one in top-5 both correct
        'avg5_joint': 0,     # Average joint correctness over top-5
        'num_queries': 0,
    }
    
    # Iterate through dataset
    for sample_idx in range(min(100, len(csv_dataset))):  # Evaluate on 100 samples for speed
        sample = csv_dataset[sample_idx]
        image_id_gt = sample['image_id']
        captions = sample['captions']
        pos_objects = sample['positive_objects']
        neg_objects = sample['negative_objects']
        
        # For each caption (query)
        for caption_idx, caption in enumerate(captions):
            # Encode query
            with torch.no_grad():
                query_tokens = clip.tokenize(caption).to(device)
                query_emb = ranking_clip.encode_text(query_tokens)
                query_emb = torch.nn.functional.normalize(query_emb, p=2, dim=-1)
            
            # Compute similarity with all images
            similarities = (query_emb @ image_embeddings.to(device).t()).squeeze()
            
            # Get top-5 images
            top5_indices = torch.topk(similarities, k=5).indices.cpu().numpy()
            top5_image_ids = [image_ids[idx] for idx in top5_indices]
            
            # Evaluate each top-5 image with MLLM
            qr_scores = []
            joint_scores = []
            
            for rank, retrieved_image_id in enumerate(top5_image_ids):
                # Get image path
                image_filename = f"{retrieved_image_id:012d}.jpg"
                image_path = coco_images_dir / image_filename
                
                if not image_path.exists():
                    continue
                
                # Question qr: Semantic correctness
                qr_score = evaluate_with_mllm(image_path, pos_objects, neg_objects, caption, 'qr')
                qr_scores.append(qr_score)
                
                # Question qn: Negation verification (only if qr passes)
                if qr_score > 0.5:
                    qn_score = evaluate_with_mllm(image_path, pos_objects, neg_objects, caption, 'qn')
                    joint_score = qr_score * qn_score  # Both must pass
                else:
                    joint_score = 0.0
                
                joint_scores.append(joint_score)
            
            if len(qr_scores) == 0:
                continue
            
            # Update metrics
            metrics['num_queries'] += 1
            
            # Top-1 metrics
            if qr_scores[0] > 0.5:
                metrics['top1_qr'] += 1
            if joint_scores[0] > 0.5:
                metrics['top1_joint'] += 1
            
            # Top-5 metrics
            if max(qr_scores) > 0.5:
                metrics['top5_qr'] += 1
            if max(joint_scores) > 0.5:
                metrics['top5_joint'] += 1
            
            # Avg-5 metrics
            metrics['avg5_qr'] += sum(qr_scores) / len(qr_scores)
            metrics['avg5_joint'] += sum(joint_scores) / len(joint_scores)
        
        if sample_idx % 10 == 0:
            print(f"  Progress: {sample_idx}/100 samples evaluated")
    
    # Normalize metrics
    if metrics['num_queries'] > 0:
        metrics['top1_qr'] /= metrics['num_queries']
        metrics['top5_qr'] /= metrics['num_queries']
        metrics['avg5_qr'] /= metrics['num_queries']
        metrics['top1_joint'] /= metrics['num_queries']
        metrics['top5_joint'] /= metrics['num_queries']
        metrics['avg5_joint'] /= metrics['num_queries']
    
    evaluation_metrics[model_name] = metrics
    
    print(f"\n✓ {model_name} Results:")
    print(f"  Semantic qr only:")
    print(f"    Top-1:  {metrics['top1_qr']:.4f}")
    print(f"    Avg-5:  {metrics['avg5_qr']:.4f}")
    print(f"    Top-5:  {metrics['top5_qr']:.4f}")
    print(f"  Joint (qr + qn):")
    print(f"    Top-1:  {metrics['top1_joint']:.4f}")
    print(f"    Avg-5:  {metrics['avg5_joint']:.4f}")
    print(f"    Top-5:  {metrics['top5_joint']:.4f}")

# Print Final Summary
print("\n" + "="*70)
print("FINAL RESULTS: Negation-Aware Image Retrieval + MLLM Judge")
print("="*70)
print(f"\n{'Model':<30} {'Top-1 qr':<12} {'Avg-5 qr':<12} {'Top-5 qr':<12} {'Top-1 joint':<12} {'Avg-5 joint':<12} {'Top-5 joint':<12}")
print("-"*110)

for model_name, metrics in evaluation_metrics.items():
    print(f"{model_name:<30} {metrics['top1_qr']:.4f}      {metrics['avg5_qr']:.4f}      {metrics['top5_qr']:.4f}      {metrics['top1_joint']:.4f}        {metrics['avg5_joint']:.4f}        {metrics['top5_joint']:.4f}")

print("="*110)



EXPERIMENT 2: NEGATION-AWARE IMAGE RETRIEVAL + MLLM JUDGE

[1/6] Loading negated retrieval CSV dataset...
✓ Loaded 100 samples

[2/6] Loading CLIP model for image ranking...
✓ CLIP ViT-B/32 loaded to cuda

[3/6] Loading and embedding COCO val2017 images...
✓ Found 5000 COCO images
Loading cached embeddings from embeddings_cache/CLIP_ViT-B32_COCO_val2017_embeddings.pt...
✓ Loaded cached embeddings for 5000 images

[4/6] Loading Qwen2-VL-2B MLLM Judge...
  Loading Qwen/Qwen2-VL-2B-Instruct (better discriminative power, ~2B parameters)...



EXPERIMENT 2: NEGATION-AWARE IMAGE RETRIEVAL + MLLM JUDGE

[1/6] Loading negated retrieval CSV dataset...
✓ Loaded 100 samples

[2/6] Loading CLIP model for image ranking...
✓ CLIP ViT-B/32 loaded to cuda

[3/6] Loading and embedding COCO val2017 images...
✓ Found 5000 COCO images
Loading cached embeddings from embeddings_cache/CLIP_ViT-B32_COCO_val2017_embeddings.pt...
✓ Loaded cached embeddings for 5000 images

[4/6] Loading Qwen2-VL-2B MLLM Judge...
  Loading Qwen/Qwen2-VL-2B-Instruct (better discriminative power, ~2B parameters)...


/tmp/ipykernel_470740/2995553147.py:56: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  embeddings_data = torch.load(cache_path)
Loading weights: 100%|██████████| 729/729 [00:


EXPERIMENT 2: NEGATION-AWARE IMAGE RETRIEVAL + MLLM JUDGE

[1/6] Loading negated retrieval CSV dataset...
✓ Loaded 100 samples

[2/6] Loading CLIP model for image ranking...
✓ CLIP ViT-B/32 loaded to cuda

[3/6] Loading and embedding COCO val2017 images...
✓ Found 5000 COCO images
Loading cached embeddings from embeddings_cache/CLIP_ViT-B32_COCO_val2017_embeddings.pt...
✓ Loaded cached embeddings for 5000 images

[4/6] Loading Qwen2-VL-2B MLLM Judge...
  Loading Qwen/Qwen2-VL-2B-Instruct (better discriminative power, ~2B parameters)...


/tmp/ipykernel_470740/2995553147.py:56: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  embeddings_data = torch.load(cache_path)
Loading weights: 100%|██████████| 729/729 [00:

✓ Qwen2-VL-2B MLLM Judge loaded (~2B parameters, better discriminative quality)

[5/6] Setting up evaluation metrics...

[6/6] Running retrieval + MLLM evaluation...

Evaluating: CLIP + MLLM Judge
  Progress: 0/100 samples evaluated
  Progress: 10/100 samples evaluated
  Progress: 20/100 samples evaluated
  Progress: 30/100 samples evaluated
  Progress: 40/100 samples evaluated
  Progress: 50/100 samples evaluated
  Progress: 60/100 samples evaluated
  Progress: 70/100 samples evaluated
  Progress: 80/100 samples evaluated
  Progress: 90/100 samples evaluated

✓ CLIP + MLLM Judge Results:
  Semantic qr only:
    Top-1:  0.8880
    Avg-5:  0.8044
    Top-5:  0.9940
  Joint (qr + qn):
    Top-1:  0.8860
    Avg-5:  0.8036
    Top-5:  0.9940

FINAL RESULTS: Negation-Aware Image Retrieval + MLLM Judge

Model                          Top-1 qr     Avg-5 qr     Top-5 qr     Top-1 joint  Avg-5 joint  Top-5 joint 
---------------------------------------------------------------------------------

In [9]:
# ============================================================================
# DEBUG: Test MLLM responses before full evaluation
# ============================================================================

print("\n" + "="*70)
print("DEBUG: Testing MLLM responses on 5 sample images")
print("="*70)

# Test on first 5 samples
for test_idx in range(min(5, len(csv_dataset))):
    sample = csv_dataset[test_idx]
    pos_objects = sample['positive_objects']
    neg_objects = sample['negative_objects']
    captions = sample['captions']
    
    print(f"\n[Sample {test_idx + 1}]")
    print(f"  Positive objects: {pos_objects}")
    print(f"  Negative objects: {neg_objects}")
    print(f"  Query: {captions[0] if captions else 'N/A'}")
    
    # Get a top-1 image
    caption = captions[0] if captions else ""
    if not caption:
        continue
    
    with torch.no_grad():
        query_tokens = clip.tokenize(caption).to(device)
        query_emb = ranking_clip.encode_text(query_tokens)
        query_emb = torch.nn.functional.normalize(query_emb, p=2, dim=-1)
    
    similarities = (query_emb @ image_embeddings.to(device).t()).squeeze()
    top1_idx = torch.topk(similarities, k=1).indices.cpu().numpy()[0]
    top1_image_id = image_ids[top1_idx]
    
    image_filename = f"{top1_image_id:012d}.jpg"
    image_path = coco_images_dir / image_filename
    
    if not image_path.exists():
        print(f"  ⚠ Image not found")
        continue
    
    # Test QR and QN prompts
    try:
        if mllm_available:
            for qtype in ['qr', 'qn']:
                img = Image.open(image_path).convert('RGB')
                
                if qtype == 'qr':
                    prompt = f"Does this image contain these objects: {', '.join(pos_objects)} but does NOT contain: {', '.join(neg_objects)}? Answer only 'yes' or 'no'."
                else:
                    prompt = f"Verify that NONE of these objects appear in the image: {', '.join(neg_objects)}. Answer only 'yes' or 'no'."
                
                conversation = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": prompt}]}]
                text_prompt = mllm_processor.apply_chat_template(conversation, add_generation_prompt=True)
                inputs = mllm_processor(text=text_prompt, images=[img], return_tensors='pt')
                inputs = {k: v.to(device) for k, v in inputs.items()}
                
                with torch.no_grad():
                    output_ids = mllm_model.generate(**inputs, max_new_tokens=50)
                
                # Extract only NEW generated tokens (skip the input prompt)
                input_token_length = inputs['input_ids'].shape[1]
                generated_token_ids = output_ids[0, input_token_length:]
                response = mllm_processor.decode(generated_token_ids, skip_special_tokens=True)
                is_yes = extract_yes_no_response(response)
                
                print(f"  [{qtype.upper()}] Response: {response[:100]}")
                print(f"         Parsed as: {'YES' if is_yes else 'NO'}")
        else:
            print(f"  MLLM not available for debugging")
    except Exception as e:
        print(f"  ⚠ Error: {e}")

print("\n" + "="*70)



DEBUG: Testing MLLM responses on 5 sample images

[Sample 1]
  Positive objects: ['chair', 'vase']
  Negative objects: ['person', 'dining table', 'cup']
  Query: The image shows a dirty table with incense burners, but no person is around.
  [QR] Response: no
         Parsed as: NO
  [QN] Response: yes
         Parsed as: YES

[Sample 2]
  Positive objects: ['person', 'toothbrush']
  Negative objects: ['chair', 'car', 'handbag']
  Query: No chair is present in the image, which shows an old picture of a baby holding a record in his hand.
  [QR] Response: no
         Parsed as: NO
  [QN] Response: yes
         Parsed as: YES

[Sample 3]
  Positive objects: ['person', 'sports ball', 'baseball bat', 'baseball glove']
  Negative objects: ['chair', 'car', 'handbag']
  Query: A man swings a bat as a ball is pitched to him, with no chair in the scene.
  [QR] Response: yes
         Parsed as: YES
  [QN] Response: yes
         Parsed as: YES

[Sample 4]
  Positive objects: ['bottle', 'sink', 'ha

## Summary and Comparison

Compare results across both experiments

In [ ]:
# ============================================================================
# SUMMARY: Compare all experiments
# ============================================================================

print("\n" + "="*70)
print("COMPREHENSIVE EXPERIMENT SUMMARY")
print("="*70)

print("\n[EXPERIMENT 1] Pairwise Preference (NegRefCOCOg Dataset)")
print("-"*70)
print("Metric: Score = 1 if sim(text, pos_image) > sim(text, neg_image)")
print("\nResults:")
for model_name, result in pairwise_results.items():
    acc = result['avg_score']
    print(f"  {model_name:<30}: {acc:.4f} ({result['correct']}/{result['total']})")

best_model_exp1 = max(pairwise_results.items(), key=lambda x: x[1]['avg_score'])
print(f"\n✓ Best Model (Exp 1): {best_model_exp1[0]} with score {best_model_exp1[1]['avg_score']:.4f}")

print("\n[EXPERIMENT 2] Recall@K (COCOValLlama Dataset)")
print("-"*70)
print("Metric: Recall@K = fraction of queries with positive in top-K")
print("\nResults:")
for model_name, recalls in recall_results.items():
    r1 = recalls.get('recall@1', 0.0)
    r5 = recalls.get('recall@5', 0.0)
    r10 = recalls.get('recall@10', 0.0)
    print(f"  {model_name:<30}")
    print(f"    Recall@1:  {r1:.4f}")
    print(f"    Recall@5:  {r5:.4f}")
    print(f"    Recall@10: {r10:.4f}")

# Find best for Recall@1
best_model_exp2 = max(recall_results.items(), key=lambda x: x[1].get('recall@1', 0.0))
print(f"\n✓ Best Model (Exp 2, Recall@1): {best_model_exp2[0]} with score {best_model_exp2[1].get('recall@1', 0.0):.4f}")

print("\n" + "="*70)
print("KEY FINDINGS")
print("="*70)
print("\n1. NegRefCOCOg (Pairwise Preference):")
print("   - Evaluates ability to distinguish positive from negative images")
print("   - Uses binary preference judgment")
print("\n2. COCOValLlama (Recall@K):")
print("   - Evaluates ranking quality of images for text queries")
print("   - Measures top-K retrieval performance")
print("\n3. Implications:")
if best_model_exp1[0] == "Baseline CLIP":
    print("   - Baseline CLIP strong on pairwise distinction")
else:
    print(f"   - {best_model_exp1[0]} superior for negation-aware matching")
print("="*70)